In [ ]:
import tensorflow as tf
import cv2 
import os
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
image_array = cv2.imread('/kaggle/input/datasets/akashshingha850/mrl-eye-dataset/data/train/sleepy/s0001_00002_0_0_0_0_0_01.png', cv2.IMREAD_GRAYSCALE)

In [ ]:
plt.imshow(image_array, cmap="gray")

In [ ]:
image_array.shape

In [ ]:
data_directory = '/kaggle/input/datasets/akashshingha850/mrl-eye-dataset/data/train'
classes = ["awake", "sleepy"]
for category in classes:
    path = os.path.join(data_directory, category)
    for image in os.listdir(path):
        image_array = cv2.imread(os.path.join(path, image), cv2.IMREAD_GRAYSCALE)
        backtorgb = cv2.cvtColor(image_array, cv2.COLOR_GRAY2RGB)
        plt.imshow(image_array, cmap="gray")
        plt.show()
        break
    break

In [ ]:
image_size = 224
new_array = cv2.resize(backtorgb, (image_size, image_size))
plt.imshow(new_array,cmap="gray")
plt.show()

reading images and converting them into an array for data and labels

In [ ]:
import random

training_data = []

def creating_training_data():
    for category in classes:
        path = os.path.join(data_directory, category)
        class_num = classes.index(category)
        
        images = os.listdir(path)
        images = random.sample(images, 1000)  # 🎯 random 1000 images
        
        for image in images:
            try:
                image_array = cv2.imread(os.path.join(path, image))
                new_array = cv2.resize(image_array, (image_size, image_size))
                training_data.append([new_array, class_num])
            except Exception as e:
                pass

In [ ]:
creating_training_data()

In [ ]:
print(len(training_data))

In [ ]:
import random

random.shuffle(training_data)

In [11]:
X = []
y = []
for features,label in training_data:
    X.append(features)
    y.append(label)
X = np.array(X).reshape(-1, image_size, image_size, 3)

In [ ]:
X.shape

Normalizing data

In [ ]:
X = X/255.0;

In [ ]:
Y = np.array(y)

In [ ]:
import pickle

pickle_out = open('/kaggle/working/X.pickle', 'wb')
pickle.dump(X, pickle_out)
pickle_out.close()

pickle_out = open('/kaggle/working/y.pickle', 'wb')
pickle.dump(y, pickle_out)
pickle_out.close()

In [ ]:
pickle_in = open('/kaggle/working/X.pickle', 'rb')
X = pickle.load(pickle_in)

pickle_in = open('/kaggle/working/y.pickle', 'rb')
y = pickle.load(pickle_in)

deep learning model for training - Transfer Learning

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
model = tf.keras.applications.mobilenet.MobileNet()

In [ ]:
model.summary()

Transfer Learning

In [ ]:
base_input = model.input

In [ ]:
base_output = model.layers[-4].output

In [ ]:
flat_layer = layers.Flatten()(base_output)
final_output = layers.Dense(1)(flat_layer)
final_output = layers.Activation('sigmoid')(final_output)

In [ ]:
new_model = keras.Model(inputs = base_input, outputs = final_output)

In [ ]:
new_model.summary()

settings for binary classification (awake/sleepy)

In [ ]:
new_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=["accuracy"])

model training

In [ ]:
model_training = new_model.fit(X, Y, epochs=16, validation_split=0.1)

Accuracy Graph

In [ ]:
plt.plot(model_training.history['accuracy'], label='Train Accuracy')
plt.plot(model_training.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Accuracy Graph')
plt.show()

Loss Graph

In [ ]:
plt.plot(model_training.history['loss'], label='Train Loss')
plt.plot(model_training.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Loss Graph')
plt.show()

Evaluation

In [ ]:
new_model.save("/kaggle/working/drowsiness_model.h5")

In [ ]:
import shutil

shutil.make_archive('output_files', 'zip', '/kaggle/working')